In [1]:
%pip install pandas matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
print(os.listdir("../data"))

['amazon_polarity_test.parquet', 'amazon_polarity_train.parquet', 'splits']


In [6]:
import pandas as pd

try:
    test_check = pd.read_parquet("../data/amazon_polarity_test.parquet")
    print("Parquet works! Shape:", test_check.shape)
except Exception as e:
    print("Parquet still blocked:", e)

Parquet works! Shape: (400000, 3)


In [7]:
train_df = pd.read_parquet("../data/amazon_polarity_train.parquet")
test_df = pd.read_parquet("../data/amazon_polarity_test.parquet")

train_df.to_csv("../data/amazon_polarity_train.csv", index=False)
test_df.to_csv("../data/amazon_polarity_test.csv", index=False)

print("Converted to CSV.")
print(train_df.shape, test_df.shape)

Converted to CSV.
(3600000, 3) (400000, 3)


In [8]:
print(os.listdir("../data"))

['amazon_polarity_test.csv', 'amazon_polarity_test.parquet', 'amazon_polarity_train.csv', 'amazon_polarity_train.parquet', 'splits']


In [9]:
import pandas as pd

train_df = pd.read_csv("../data/amazon_polarity_train.csv")
test_df = pd.read_csv("../data/amazon_polarity_test.csv")

print(train_df.shape)
print(test_df.shape)

(3600000, 3)
(400000, 3)


In [10]:
train_sample = train_df.sample(n=20000, random_state=42).reset_index(drop=True)
print(train_sample.shape)

(20000, 3)


In [11]:
print(train_sample["label"].value_counts())
print(train_sample["label"].value_counts(normalize=True) * 100)

label
1    10027
0     9973
Name: count, dtype: int64
label
1    50.135
0    49.865
Name: proportion, dtype: float64


In [7]:
train_sample["content_length"] = train_sample["content"].str.len()
print(train_sample["content_length"].describe())

count    20000.000000
mean       403.874050
std        233.275741
min         50.000000
25%        206.000000
50%        355.000000
75%        563.000000
max       1003.000000
Name: content_length, dtype: float64


In [8]:
max_len = train_sample["content_length"].max()
candidate_edges = [0, 100, 200, 300, 500, 750, 1000, 1500, 2000]
bins = [b for b in candidate_edges if b < max_len] + [max_len + 1]
labels = [f"{bins[i]}-{bins[i+1]-1}" for i in range(len(bins)-1)]

bucketed = pd.cut(train_sample["content_length"], bins=bins, labels=labels)
print(bucketed.value_counts().sort_index())

content_length
0-99          432
100-199      4354
200-299      3551
300-499      5306
500-749      4155
750-999      2201
1000-1003       1
Name: count, dtype: int64


In [9]:
print("Shortest reviews:")
print(train_sample.nsmallest(5, "content_length")[["label", "content"]])

print("\nLongest reviews:")
print(train_sample.nlargest(5, "content_length")[["label", "content"]])

Shortest reviews:
       label                                            content
8395       0  I expected more from a s/w marketed by Broderb...
11919      1  Loved it. Highly suggest it. Couldn't put it d...
12126      1  this book was full of fun but it wasn't very l...
18901      1  I THINK THE VIDEO IS VERY PHAT I WISH I COULD ...
19043      0  this is the worst SVH book i have ever read in...

Longest reviews:
       label                                            content
12646      0  I own 38 U2 CDs, and this is the worst one. I ...
7460       0  I guess you get what you pay for, I was not fa...
14611      1  Here's the premise - you're this little guy wi...
19852      1  This book was an epiphany when I encountered i...
6110       1  I had to read this Book for my Operation Manag...


In [10]:
print("Exact duplicate reviews:", train_sample["content"].duplicated().sum())
print("Empty/whitespace-only reviews:", (train_sample["content"].str.strip() == "").sum())

Exact duplicate reviews: 0
Empty/whitespace-only reviews: 0


In [11]:
ambiguous_check = train_sample.sample(10, random_state=7)
for _, row in ambiguous_check.iterrows():
    print(f"Label: {row['label']} | {row['content'][:250]}")
    print("-" * 80)

Label: 1 | I saw this movie one day when there was nothing else to watch. I must admit I was slightly disturbed by Macaulay Culkins character. He was bizarre and his actions appalling for a twelve year old. On the other hand, Elijahs performance was the relief 
--------------------------------------------------------------------------------
Label: 1 | To dissolbe the doubts of others, I would like to say that this book is a tribute to LoR. But if you think about it, arent' most books or movies a tribute to something. Every type of writing has a basic idea that most books follow.This book is a non-
--------------------------------------------------------------------------------
Label: 1 | Billy Boyle is an unlikely hero. He isn't on Uncle Ike's team because he's that good or that brave. He's there because he took advantage of a distant relationship to avoid going to the front lines. Only to find out that friendly Uncle Ike isn't going
----------------------------------------------------

## Day 2 Findings
- Dataset is pre-balanced (~50/50 positive/negative), confirmed via value_counts
- Review length: max [X] characters, median ~[Y], right-skewed
- No significant duplicates or empty reviews found in a 20K sample
- Some ambiguous labels found on manual review — expect some irreducible label noise